# KDAL Station Stacking V20 NBM-Physics Fix

KDAL-only patch retaining NBM temperature timing and HRRR solar/cloud/precipitation physics while excluding HRRR temperature-curve features. The final ridge prediction receives a capped monthly correction learned only from forward OOF stack residuals. This provisional run evaluates through July 1, 2026.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KDAL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v20_kdal_nbm_physics_stack"
EXPORT_MODEL_WEIGHTS = False
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V20_KDAL_NBM_PHYSICS_ENGINEERED_FEATURE_COLUMNS,
    V20_KDAL_NBM_PHYSICS_RAW_FEATURE_COLUMNS,
    kdal_oof_residual_calibrated_stack_predictions,
    V20_ENGINEERED_FEATURE_COLUMNS,
    V20_PEAK_TIMING_RAW_FEATURE_COLUMNS,
    build_station_wide_dataset,
    v20_peak_timing_readiness,
    V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS,
    _fit_feature_columns,
    _modeling_frame,
    V11_DROPPED_FEATURE_COLUMNS,
    V11_FEATURE_COLUMNS,
    V20_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V11 Contract

`feature_version="v20_kdal_nbm_physics"` keeps the v9 feature contract and remaining-warmup target, but trains base learners with Huber-style objectives while retaining the ridge stack selected by validation MAE.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in V20_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_to_2022,2021,2021,2022
1,fold_2021_2022_to_2023,2021,2022,2023
2,fold_2021_2023_to_2024,2021,2023,2024
3,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V11_FEATURE_COLUMNS, sorted(V11_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
6,KDAL,gfs,1993,2021-01-01,2026-07-12
7,KDAL,hrrr,1999,2021-01-01,2026-07-12
8,KDAL,nbm,1998,2021-01-01,2026-07-12


## Model Scores


## Peak-Timing and Wunderground Readiness Gate


In [6]:
readiness_features = build_station_wide_dataset(
    PROJECT_ROOT,
    station_id=STATION_ID,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    feature_version="v20_kdal_nbm_physics",
    target_source="wunderground_only",
)
EVALUATION_END_DATE = "2026-07-01"  # provisional; official V20 cutoff remains 2026-07-14
v20_ready, readiness_summary, readiness_missing_dates, readiness_fold_missingness = v20_peak_timing_readiness(
    readiness_features,
    station_id=STATION_ID,
    folds=V20_EXPANDING_FOLDS,
    max_missing_fraction=0.03,
    end_date=EVALUATION_END_DATE,
)
readiness_dir = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v20_kdal_fix"
readiness_dir.mkdir(parents=True, exist_ok=True)
readiness_summary.to_csv(readiness_dir / f"{STATION_ID}_readiness_summary.csv", index=False)
readiness_missing_dates.to_csv(readiness_dir / f"{STATION_ID}_readiness_missing_dates.csv", index=False)
readiness_fold_missingness.to_csv(readiness_dir / f"{STATION_ID}_readiness_fold_feature_missingness.csv", index=False)
display(readiness_summary)
display(readiness_fold_missingness.groupby(["fold", "retained"], as_index=False).agg(feature_count=("feature", "nunique")))
if not v20_ready:
    raise RuntimeError(
        "V20 readiness failed. Audit artifacts were written; rerun this notebook after peak-timing and "
        "Wunderground pulls reduce each station-year missing fraction to 3% or less."
    )


D:\dev\weather-research\src\calibration\station_stacking.py:3540: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:3539: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:3540: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,station_id,year,expected_days,peak_ready_days,wunderground_target_days,peak_missing_fraction,target_missing_fraction,peak_ready,target_ready
0,KDAL,2021,365,364,364,0.00274,0.002740,True,True
1,KDAL,2022,365,365,364,0.00000,0.002740,True,True
2,KDAL,2023,365,365,364,0.00000,0.002740,True,True
3,KDAL,2024,366,366,365,0.00000,0.002732,True,True
4,KDAL,2025,365,365,364,0.00000,0.002740,True,True
5,KDAL,2026,182,182,181,0.00000,0.005495,True,True


,fold,retained,feature_count
0,fold_2021_2022_to_2023,False,1
1,fold_2021_2022_to_2023,True,54
2,fold_2021_2023_to_2024,False,1
3,fold_2021_2023_to_2024,True,54
4,fold_2021_2024_to_2025,False,1
5,fold_2021_2024_to_2025,True,54
6,fold_2021_to_2022,False,1
7,fold_2021_to_2022,True,54
8,test_refit_2021_2025,False,1
9,test_refit_2021_2025,True,54


In [7]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v20_kdal_nbm_physics",
    target_mode="remaining_warmup",
    target_source="wunderground_only",
    max_feature_missing_fraction=0.03,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=V20_EXPANDING_FOLDS,
    year_split_validation_weights={2022: 1.0, 2023: 1.0, 2024: 1.0, 2025: 1.0},
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v20_kdal_fix",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v20_kdal_fix/KDAL_optuna.sqlite3')

In [8]:
result = run_station_year_split_experiment(config)
result.scoreboard


D:\dev\weather-research\src\calibration\station_stacking.py:3540: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:3539: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:3540: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

[I 2026-07-16 10:54:10,792] A new study created in RDB with name: KDAL_v20_kdal_nbm_physics_remaining_warmup_base_xgboost_mae_f_wide


[I 2026-07-16 10:54:14,138] Trial 0 finished with value: 1.6440167185606287 and parameters: {'n_estimators': 1342, 'learning_rate': 0.19043899115982607, 'max_depth': 9, 'min_child_weight': 2.481040974867813, 'gamma': 2.340279606636548, 'subsample': 0.4513964382185317, 'colsample_bytree': 0.3877543479093296, 'reg_alpha': 2.478071022662141, 'reg_lambda': 0.6132587025321562}. Best is trial 0 with value: 1.6440167185606287.


[I 2026-07-16 10:54:20,748] Trial 1 finished with value: 2.2713010804927554 and parameters: {'n_estimators': 2493, 'learning_rate': 0.001120367191095075, 'max_depth': 12, 'min_child_weight': 21.368329072358772, 'gamma': 3.185086660174142, 'subsample': 0.4681862286846154, 'colsample_bytree': 0.46921293140473197, 'reg_alpha': 4.476173538513514e-07, 'reg_lambda': 0.20253776634919213}. Best is trial 0 with value: 1.6440167185606287.


[I 2026-07-16 10:54:24,815] Trial 2 finished with value: 1.454956015945798 and parameters: {'n_estimators': 1540, 'learning_rate': 0.004992867109217114, 'max_depth': 8, 'min_child_weight': 0.03613894271216528, 'gamma': 4.3821697280282725, 'subsample': 0.5881351981408995, 'colsample_bytree': 0.6464454897410734, 'reg_alpha': 0.26430874752361627, 'reg_lambda': 0.0018119678627933364}. Best is trial 2 with value: 1.454956015945798.


...

C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-

C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-

C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-

C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Faris Fadil Arifin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,1452,1.444570,1.937097
1,validation_2024_2025,lightgbm,1452,1.403822,1.902358
2,validation_2024_2025,catboost,1452,1.406624,1.904975
3,validation_2024_2025,provider_mean,1452,2.717663,3.648723
4,validation_2024_2025,provider_median,1452,2.344602,3.323749
5,validation_2024_2025,nbm_raw,1452,2.078083,2.977793
6,validation_2024_2025,hrrr_raw,1452,4.924938,6.153463
7,validation_2024_2025,gfs_raw,1452,3.041998,4.098564
8,test_2026,xgboost,170,1.393122,1.808521
9,test_2026,lightgbm,170,1.377104,1.806385


## KDAL Forward-OOF Residual Calibration


In [9]:
calibrated_predictions, residual_calibration, calibration_oof_predictions = (
    kdal_oof_residual_calibrated_stack_predictions(
        result.validation_predictions,
        result.test_predictions,
        result.stack_tuning_results,
        config,
        min_month_rows=60,
        shrinkage_rows=60,
        correction_cap_f=0.75,
    )
)
if calibrated_predictions.empty:
    raise RuntimeError("KDAL OOF residual calibration produced no test predictions.")

calibrated_predictions.to_csv(
    config.resolved_output_dir() / f"{STATION_ID}_oof_calibrated_test_predictions.csv", index=False
)
residual_calibration.to_csv(
    config.resolved_output_dir() / f"{STATION_ID}_oof_residual_calibration.csv", index=False
)
calibration_oof_predictions.to_csv(
    config.resolved_output_dir() / f"{STATION_ID}_stack_calibration_oof_predictions.csv", index=False
)
residual_calibration


,month,month_rows,month_mean_residual_f,global_mean_residual_f,correction_f,selected_param_key,correction_cap_f,min_month_rows,shrinkage_rows
0,1,92,-0.011189,0.158317,0.055721,stack_trial_27,0.75,60,60
1,2,85,0.454676,0.158317,0.332045,stack_trial_27,0.75,60,60
2,3,90,-0.090934,0.158317,0.008767,stack_trial_27,0.75,60,60
3,4,90,-0.439850,0.158317,-0.200583,stack_trial_27,0.75,60,60
4,5,93,0.120205,0.158317,0.135151,stack_trial_27,0.75,60,60
5,6,90,0.038046,0.158317,0.086155,stack_trial_27,0.75,60,60
6,7,93,0.441614,0.158317,0.330517,stack_trial_27,0.75,60,60
7,8,93,0.608301,0.158317,0.431837,stack_trial_27,0.75,60,60
8,9,90,0.603168,0.158317,0.425228,stack_trial_27,0.75,60,60
9,10,93,-0.110704,0.158317,-0.005206,stack_trial_27,0.75,60,60


In [10]:
calibrated_error = pd.to_numeric(calibrated_predictions["error_f"], errors="coerce")
calibrated_absolute_error = pd.to_numeric(calibrated_predictions["absolute_error_f"], errors="coerce")
calibrated_metrics = pd.DataFrame([
    {
        "method": "ridge_stack_oof_calibrated",
        "count": int(calibrated_absolute_error.notna().sum()),
        "mae_f": float(calibrated_absolute_error.mean()),
        "rmse_f": float(np.sqrt(np.mean(np.square(calibrated_error.dropna())))),
        "bias_f": float(calibrated_error.mean()),
        "p95_absolute_error_f": float(calibrated_absolute_error.quantile(0.95)),
        "large_error_5f_pct": float(calibrated_absolute_error.ge(5.0).mean() * 100.0),
        "within_1f_pct": float(calibrated_absolute_error.le(1.0).mean() * 100.0),
        "within_2f_pct": float(calibrated_absolute_error.le(2.0).mean() * 100.0),
        "within_3f_pct": float(calibrated_absolute_error.le(3.0).mean() * 100.0),
    }
])
calibrated_metrics.to_csv(
    config.resolved_output_dir() / f"{STATION_ID}_oof_calibrated_metrics.csv", index=False
)
calibrated_metrics


,method,count,mae_f,rmse_f,bias_f,p95_absolute_error_f,large_error_5f_pct,within_1f_pct,within_2f_pct,within_3f_pct
0,ridge_stack_oof_calibrated,170,1.358732,1.787453,0.26462,3.708212,1.764706,51.176471,74.705882,90.588235


In [11]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        source_pipeline="notebooks/station_stacking_v20_kdal_fix",
    )

    exported_weights.bundle_path, exported_weights.manifest_path
else:
    print("Model export disabled for this experimental notebook.")


Model export disabled for this experimental notebook.


## V11 Feature Coverage


In [12]:
v11_feature_coverage = (
    result.features[V11_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v11_feature_coverage


,feature,coverage_pct
0,v4_observed_precip_any,100.000000
1,v4_forecast_wet_observed_dry,100.000000
2,v4_forecast_observed_precip_match,100.000000
3,v4_all_forecast_precip,100.000000
4,v4_any_forecast_precip,100.000000
5,climatology_high_10y_std_f,100.000000
6,climatology_high_10y_f,100.000000
7,climatology_high_10y_count,100.000000
8,v8_month_remaining_warmup_count,100.000000
9,v4_observed_wet_forecast_dry,100.000000


In [13]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V11_FEATURE_COLUMNS)]


,feature,kind
3,observed_temp_change_last_1h_f,numeric
4,observed_temp_change_last_3h_f,numeric
5,observed_morning_warmup_rate_f_per_hour,numeric
6,observed_high_so_far_change_since_9am_f,numeric
26,v2_recent_heat_anomaly_f,numeric
27,v2_recent_heat_momentum_f,numeric
28,v2_morning_warmup_to_consensus_f,numeric
29,v2_consensus_minus_7d_actual_f,numeric
30,v2_spread_per_warmup_f,numeric
31,v2_humidity_warmup_interaction,numeric


## Dropped Feature Check


In [14]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V11_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [15]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,98.91143
1,observed_temp_change_last_3h_f,98.91143
2,observed_morning_warmup_rate_f_per_hour,98.91143
3,observed_high_so_far_change_since_9am_f,98.91143


## Rounded Within 1F Accuracy


In [16]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
0,oof_2026,catboost,170,108,63.529412
7,oof_2026,ridge_stack,170,108,63.529412
3,oof_2026,lightgbm,170,104,61.176471
8,oof_2026,xgboost,170,103,60.588235
4,oof_2026,nbm_raw,170,84,49.411765
1,oof_2026,gfs_raw,170,69,40.588235
6,oof_2026,provider_median,170,69,40.588235
5,oof_2026,provider_mean,170,57,33.529412
2,oof_2026,hrrr_raw,170,17,10.000000
9,validation_2024_2025,catboost,1452,920,63.360882


## Version Comparison


In [17]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
    ("v11", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,ridge_stack,138,1.369446,1.890651,v9
1,test_2026,xgboost,138,1.402066,1.889472,v11
2,test_2026,xgboost,138,1.419940,1.895994,v9
3,test_2026,ridge_stack,138,1.429989,1.907881,v11
4,test_2026,lightgbm,138,1.454606,1.932482,v11
...,...,...,...,...,...,...
83,validation_2024_2025,hrrr_raw,660,5.477361,6.266007,v3
84,validation_2024_2025,hrrr_raw,668,5.486824,6.274688,v7
85,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v5
86,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v6


## 2026 OOF Weather Brackets


In [18]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bucket_log_loss,bracket_accuracy_pct,p95_absolute_error_f,large_miss_5f_pct
0,xgboost,170,1.393122,1.808521,1.372603,43.529412,3.435454,1.176471
1,lightgbm,170,1.377104,1.806385,1.368934,45.294118,3.656903,1.176471
2,catboost,170,1.417672,1.864892,1.400749,43.529412,3.712265,2.352941
3,ridge_stack,170,1.371846,1.800089,1.356020,42.941176,3.742628,1.764706
4,provider_mean,170,2.583543,3.303107,1.678281,23.529412,5.481044,8.235294
5,provider_median,170,2.428105,3.214617,1.757808,22.941176,5.950489,8.823529
6,nbm_raw,170,2.045105,2.795566,1.726940,34.117647,5.718984,8.235294
7,hrrr_raw,170,4.927253,5.592931,1.838395,5.294118,9.733436,45.294118
8,gfs_raw,170,2.550131,3.455843,1.948382,27.058824,6.385804,11.176471


## Train-Fold 3% Missingness Audit


In [19]:
modeling_frame, candidate_categorical, candidate_numeric = _modeling_frame(result.features, config)
candidate_features = [*candidate_categorical, *candidate_numeric]
audit_specs = [
    (fold.name, fold.train_start_year, fold.train_end_year)
    for fold in V20_EXPANDING_FOLDS
] + [("test_refit_2021_2025", 2021, 2025)]

missingness_rows = []
years = pd.to_numeric(modeling_frame["year"], errors="coerce")
for fold_name, train_start, train_end in audit_specs:
    train = modeling_frame.loc[years.between(train_start, train_end)].copy()
    retained_categorical, retained_numeric = _fit_feature_columns(
        train,
        candidate_categorical,
        candidate_numeric,
        max_missing_fraction=config.effective_max_feature_missing_fraction,
    )
    retained = set(retained_categorical) | set(retained_numeric)
    for feature in candidate_features:
        numeric_feature = feature in candidate_numeric
        values = pd.to_numeric(train[feature], errors="coerce") if numeric_feature else train[feature]
        missingness_rows.append(
            {
                "fold": fold_name,
                "train_start_year": train_start,
                "train_end_year": train_end,
                "feature": feature,
                "kind": "numeric" if numeric_feature else "categorical",
                "missing_fraction": float(values.isna().mean()),
                "retained": feature in retained,
            }
        )

fold_feature_missingness = pd.DataFrame(missingness_rows)
retained_dropped_summary = (
    fold_feature_missingness.groupby(["fold", "retained"], as_index=False)
    .agg(feature_count=("feature", "nunique"), maximum_missing_fraction=("missing_fraction", "max"))
)
fold_feature_missingness.to_csv(config.resolved_output_dir() / f"{STATION_ID}_fold_feature_missingness.csv", index=False)
retained_dropped_summary, fold_feature_missingness.loc[~fold_feature_missingness["retained"]].sort_values(
    ["fold", "missing_fraction"], ascending=[True, False]
)


(                     fold  retained  feature_count  maximum_missing_fraction
 0  fold_2021_2022_to_2023     False             24                  0.915278
 1  fold_2021_2022_to_2023      True             76                  0.006944
 2  fold_2021_2023_to_2024     False             22                  0.907664
 3  fold_2021_2023_to_2024      True             78                  0.022161
 4  fold_2021_2024_to_2025     False             22                  0.911541
 5  fold_2021_2024_to_2025      True             78                  0.016586
 6       fold_2021_to_2022     False             24                  0.896936
 7       fold_2021_to_2022      True             76                  0.013928
 8    test_refit_2021_2025     False             22                  0.915516
 9    test_refit_2021_2025      True             78                  0.013252,
                        fold  train_start_year  train_end_year  \
 101  fold_2021_2022_to_2023              2021            2022   
 168  fol

## Expanded 11 AM Feature Coverage and Provider Count


In [20]:
new_feature_coverage = (
    result.features[V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)
provider_count_coverage = (
    result.features["v11sf_forecast_temp_11am_provider_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("available_provider_count")
    .reset_index(name="row_count")
)
provider_count_coverage["row_pct"] = provider_count_coverage["row_count"] / len(result.features) * 100
new_feature_coverage.to_csv(config.resolved_output_dir() / f"{STATION_ID}_11am_feature_coverage.csv", index=False)
new_feature_coverage, provider_count_coverage


(                                              feature  coverage_pct
 0                     v11sf_forecast_temp_11am_mean_f      98.91143
 1                   v11sf_forecast_temp_11am_median_f      98.91143
 2           v11sf_forecast_temp_11am_minus_observed_f      98.91143
 3                v11sf_forecast_temp_11am_abs_error_f      98.91143
 4               v11sf_forecast_temp_11am_warm_error_f      98.91143
 5               v11sf_forecast_temp_11am_cool_error_f      98.91143
 6                   v11sf_forecast_temp_11am_spread_f      98.91143
 7             v11sf_forecast_temp_11am_provider_count     100.00000
 8   v11sf_forecast_temp_bias_remaining_warmup_inte...      98.91143
 9          v11sf_observation_adjusted_provider_high_f      98.91143
 10                 v11sf_forecast_warmup_after_11am_f      98.91143,
    available_provider_count  row_count    row_pct
 0                         0         22   1.088570
 1                         1         23   1.138050
 2                

## New-Feature Importance


In [21]:
new_feature_importance = result.feature_importance.loc[
    result.feature_importance["feature"].isin(V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS)
].sort_values(["method", "importance_mean_mae_f"], ascending=[True, False])
new_feature_importance


,method,param_key,feature,importance_mean_mae_f,importance_std_mae_f,n_repeats,train_start_year,train_end_year,test_year,train_rows,test_rows
21,catboost,trial_20,v11sf_forecast_temp_11am_median_f,0.050684,0.022572,10,2021,2025,2026,1811,170
29,catboost,trial_20,v11sf_observation_adjusted_provider_high_f,0.032357,0.020129,10,2021,2025,2026,1811,170
46,catboost,trial_20,v11sf_forecast_warmup_after_11am_f,0.017596,0.016837,10,2021,2025,2026,1811,170
64,catboost,trial_20,v11sf_forecast_temp_11am_minus_observed_f,0.009965,0.003971,10,2021,2025,2026,1811,170
67,catboost,trial_20,v11sf_forecast_temp_11am_mean_f,0.009887,0.007188,10,2021,2025,2026,1811,170
77,catboost,trial_20,v11sf_forecast_temp_11am_spread_f,0.007513,0.004547,10,2021,2025,2026,1811,170
93,catboost,trial_20,v11sf_forecast_temp_bias_remaining_warmup_inte...,0.004768,0.006900,10,2021,2025,2026,1811,170
95,catboost,trial_20,v11sf_forecast_temp_11am_warm_error_f,0.004589,0.003834,10,2021,2025,2026,1811,170
160,catboost,trial_20,v11sf_forecast_temp_11am_provider_count,0.000000,0.000000,10,2021,2025,2026,1811,170
214,catboost,trial_20,v11sf_forecast_temp_11am_abs_error_f,-0.002478,0.005585,10,2021,2025,2026,1811,170


## 2026 Monthly Metrics


In [22]:
monthly_predictions = result.test_predictions.copy()
monthly_predictions["month"] = pd.to_datetime(monthly_predictions["contract_date"], errors="coerce").dt.month
monthly_metrics = (
    monthly_predictions.dropna(subset=["month", "error_f"])
    .groupby(["method", "month"], as_index=False)
    .agg(
        count=("error_f", "size"),
        mae_f=("absolute_error_f", "mean"),
        rmse_f=("error_f", lambda values: float(np.sqrt(np.mean(np.square(values))))),
        bias_f=("error_f", "mean"),
    )
)
monthly_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_2026_monthly_metrics.csv", index=False)
monthly_metrics


,method,month,count,mae_f,rmse_f,bias_f
0,catboost,1,31,1.270353,1.745002,0.350432
1,catboost,2,28,1.279802,1.487835,0.244155
2,catboost,3,30,1.857212,2.265888,0.669492
3,catboost,4,29,1.170218,1.598603,0.033795
4,catboost,5,31,1.470230,2.118021,-0.039316
5,catboost,6,21,1.455194,1.794795,-0.103470
6,gfs_raw,1,31,2.272084,2.918034,1.454306
7,gfs_raw,2,28,2.593110,3.145965,2.344093
8,gfs_raw,3,30,2.316502,3.900639,1.927191
9,gfs_raw,4,29,2.979966,3.917502,1.035080


## Performance by Warm/Cool 11 AM Forecast Delta


In [23]:
delta_by_date = result.features[[
    "contract_date",
    "v11sf_forecast_temp_11am_minus_observed_f",
]].copy()
delta_predictions = result.test_predictions.merge(delta_by_date, on="contract_date", how="left")
delta_predictions["forecast_temp_delta_bucket"] = pd.cut(
    delta_predictions["v11sf_forecast_temp_11am_minus_observed_f"],
    bins=[-np.inf, -2.0, -0.5, 0.5, 2.0, np.inf],
    labels=["cool_gt_2f", "cool_0.5_to_2f", "near_match", "warm_0.5_to_2f", "warm_gt_2f"],
)
warm_cool_metrics = (
    delta_predictions.dropna(subset=["forecast_temp_delta_bucket", "error_f"])
    .groupby(["method", "forecast_temp_delta_bucket"], observed=True, as_index=False)
    .agg(count=("error_f", "size"), mae_f=("absolute_error_f", "mean"), bias_f=("error_f", "mean"))
)
warm_cool_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_warm_cool_delta_metrics.csv", index=False)
warm_cool_metrics


,method,forecast_temp_delta_bucket,count,mae_f,bias_f
0,catboost,cool_gt_2f,44,1.579404,0.230000
1,catboost,cool_0.5_to_2f,61,1.291800,0.365971
2,catboost,near_match,30,1.346123,0.111178
3,catboost,warm_0.5_to_2f,27,1.440414,0.409092
4,catboost,warm_gt_2f,8,1.679477,-1.431515
5,gfs_raw,cool_gt_2f,44,2.320652,1.990974
6,gfs_raw,cool_0.5_to_2f,61,2.575091,0.574414
7,gfs_raw,near_match,30,1.737724,0.214046
8,gfs_raw,warm_0.5_to_2f,27,2.405779,0.065448
9,gfs_raw,warm_gt_2f,8,7.155651,-1.565332


## Common-Date Comparison with Existing V11 Settlement Fix


In [24]:
baseline_path = (
    PROJECT_ROOT
    / "data"
    / "calibration"
    / "station_stacking_v11_settlement_fix"
    / f"{STATION_ID}_year_split_test_predictions.csv"
)
baseline_predictions = pd.read_csv(baseline_path)
baseline_predictions["contract_date"] = baseline_predictions["contract_date"].astype(str).str[:10]
v20_predictions = result.test_predictions.copy()
v20_predictions["contract_date"] = v20_predictions["contract_date"].astype(str).str[:10]
comparison = baseline_predictions.merge(
    v20_predictions,
    on=["contract_date", "method"],
    suffixes=("_baseline", "_v20"),
)
comparison["baseline_abs_error_f"] = (
    pd.to_numeric(comparison["actual_high_f_baseline"], errors="coerce")
    - pd.to_numeric(comparison["predicted_high_f_baseline"], errors="coerce")
).abs()
comparison["v20_abs_error_f"] = (
    pd.to_numeric(comparison["actual_high_f_v20"], errors="coerce")
    - pd.to_numeric(comparison["predicted_high_f_v20"], errors="coerce")
).abs()
common_date_comparison = (
    comparison.groupby("method", as_index=False)
    .agg(
        common_date_count=("contract_date", "size"),
        baseline_mae_f=("baseline_abs_error_f", "mean"),
        v20_mae_f=("v20_abs_error_f", "mean"),
        v20_better_days=("v20_abs_error_f", lambda values: int((values < comparison.loc[values.index, "baseline_abs_error_f"]).sum())),
        baseline_better_days=("v20_abs_error_f", lambda values: int((values > comparison.loc[values.index, "baseline_abs_error_f"]).sum())),
    )
)
common_date_comparison["v20_delta_mae_f"] = common_date_comparison["v20_mae_f"] - common_date_comparison["baseline_mae_f"]
common_date_comparison.to_csv(config.resolved_output_dir() / f"{STATION_ID}_v11_common_date_comparison.csv", index=False)
common_date_comparison.sort_values("v20_delta_mae_f")


,method,common_date_count,baseline_mae_f,v20_mae_f,v20_better_days,baseline_better_days,v20_delta_mae_f
1,gfs_raw,170,2.550131,2.550131,0,0,0.000000
2,hrrr_raw,170,4.927253,4.927253,0,0,0.000000
5,provider_mean,170,2.583543,2.583543,2,6,0.000000
4,nbm_raw,170,2.045105,2.045105,0,0,0.000000
6,provider_median,170,2.428105,2.428105,0,0,0.000000
3,lightgbm,170,1.344043,1.377104,75,91,0.033062
8,xgboost,170,1.326717,1.393122,85,82,0.066404
7,ridge_stack,170,1.300208,1.371846,70,100,0.071638
0,catboost,170,1.343391,1.417672,64,102,0.074281


## V20 Peak-Timing Feature Coverage


In [25]:
v20_feature_columns = [*V20_PEAK_TIMING_RAW_FEATURE_COLUMNS, *V20_ENGINEERED_FEATURE_COLUMNS]
v20_feature_coverage = (
    result.features[v20_feature_columns]
    .notna().mean().mul(100)
    .rename("coverage_pct").reset_index().rename(columns={"index": "feature"})
)
v20_feature_coverage.to_csv(
    config.resolved_output_dir() / f"{STATION_ID}_v20_peak_feature_coverage.csv", index=False
)
v20_feature_coverage.sort_values("coverage_pct")


,feature,coverage_pct
5,nbm_cooling_onset_hour_local,92.083127
30,v20_nbm_observation_adjusted_high_f,98.861950
26,v20_nbm_t11_minus_observed_f,98.861950
25,v20_hrrr_t11_minus_observed_f,98.911430
29,v20_hrrr_observation_adjusted_high_f,98.911430
31,v20_adjusted_high_mean_f,98.911430
32,v20_adjusted_high_spread_f,98.911430
2,nbm_hour_of_max_local,99.950520
17,hrrr_precip_wet_hours_11_to_nbm_peak,99.950520
22,hrrr_tcc_11_to_nbm_peak_max_pct,99.950520


## Fold Metrics and Peak-Feature Importance


In [26]:
fold_metrics = (
    result.validation_predictions.groupby(["fold", "method"], as_index=False)
    .agg(count=("absolute_error_f", "size"), mae_f=("absolute_error_f", "mean"), bias_f=("error_f", "mean"))
)
peak_feature_importance = result.feature_importance.loc[
    result.feature_importance["feature"].isin(v20_feature_columns)
].sort_values(["method", "importance_mean_mae_f"], ascending=[True, False])
fold_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_fold_metrics.csv", index=False)
fold_metrics, peak_feature_importance


(                      fold           method  count     mae_f    bias_f
 0   fold_2021_2022_to_2023         catboost    363  1.526815 -0.625061
 1   fold_2021_2022_to_2023          gfs_raw    363  3.028109 -0.132348
 2   fold_2021_2022_to_2023         hrrr_raw    363  5.080056  4.869236
 3   fold_2021_2022_to_2023         lightgbm    363  1.443462 -0.656240
 4   fold_2021_2022_to_2023          nbm_raw    363  2.039734  0.963740
 5   fold_2021_2022_to_2023    provider_mean    363  2.672864  1.900209
 6   fold_2021_2022_to_2023  provider_median    363  2.277639  1.410090
 7   fold_2021_2022_to_2023          xgboost    363  1.465844 -0.630829
 8   fold_2021_2023_to_2024         catboost    364  1.229084  0.030233
 9   fold_2021_2023_to_2024          gfs_raw    364  2.468857  0.110129
 10  fold_2021_2023_to_2024         hrrr_raw    364  4.475751  4.295993
 11  fold_2021_2023_to_2024         lightgbm    364  1.283255 -0.238633
 12  fold_2021_2023_to_2024          nbm_raw    364  1.880132  1

## Calibrated Common-Date Comparison with KDAL V11 Settlement Fix


In [27]:
v11_fix_path = (
    PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11_settlement_fix"
    / f"{STATION_ID}_year_split_test_predictions.csv"
)
v11_fix = pd.read_csv(v11_fix_path)
v11_fix = v11_fix.loc[v11_fix["method"].eq("ridge_stack")].copy()
v11_fix["contract_date"] = v11_fix["contract_date"].astype(str).str[:10]
candidate = calibrated_predictions.copy()
candidate["contract_date"] = candidate["contract_date"].astype(str).str[:10]
common = candidate.merge(v11_fix, on="contract_date", suffixes=("_candidate", "_v11_fix"))
common["candidate_absolute_error_f"] = (
    pd.to_numeric(common["actual_high_f_candidate"], errors="coerce")
    - pd.to_numeric(common["predicted_high_f_candidate"], errors="coerce")
).abs()
common["v11_fix_absolute_error_f"] = (
    pd.to_numeric(common["actual_high_f_v11_fix"], errors="coerce")
    - pd.to_numeric(common["predicted_high_f_v11_fix"], errors="coerce")
).abs()
common_date_comparison = pd.DataFrame([
    {
        "common_date_count": len(common),
        "candidate_mae_f": common["candidate_absolute_error_f"].mean(),
        "v11_fix_mae_f": common["v11_fix_absolute_error_f"].mean(),
        "delta_mae_f": common["candidate_absolute_error_f"].mean() - common["v11_fix_absolute_error_f"].mean(),
        "candidate_better_days": int((common["candidate_absolute_error_f"] < common["v11_fix_absolute_error_f"]).sum()),
        "v11_fix_better_days": int((common["candidate_absolute_error_f"] > common["v11_fix_absolute_error_f"]).sum()),
    }
])
common_date_comparison.to_csv(
    config.resolved_output_dir() / f"{STATION_ID}_calibrated_v11_fix_common_date_comparison.csv", index=False
)
common_date_comparison


,common_date_count,candidate_mae_f,v11_fix_mae_f,delta_mae_f,candidate_better_days,v11_fix_better_days
0,170,1.358732,1.300208,0.058524,69,101
